# Architecture Benchmark: MobileBERT vs TinyBERT-4 vs ELECTRA-small

**Phase 2 of CanaryOS Text Classification Research**

Trains three student architectures on the Phase 1 synthetic dataset using identical hyperparameters,
and evaluates each on the real-world holdout to produce F1/precision/recall metrics.

**Candidates:**
- MobileBERT (24.6M params, 24 layers, hidden=512)
- TinyBERT-4 (14.4M params, 4 layers, hidden=312)
- ELECTRA-small (13.5M params, 12 layers, hidden=256)

**Excluded:** DistilBERT (66M params) -- exceeds 50MB INT8 budget

**Protocol (D-04/D-05):** Fixed hyperparameters, 5 epochs, lr=2e-5. This is a ranking exercise, not optimization.

**Primary metric (D-07):** F1 on real-world holdout (`holdout_realworld.jsonl`)

In [ ]:
# Cell 1: Environment setup
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"  # MUST be before torch import

import torch
import json
import time
import numpy as np
from pathlib import Path
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)
from datasets import Dataset
from sklearn.metrics import (
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)
import matplotlib.pyplot as plt

# Device check
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")
assert device == "mps" or device == "cpu", f"Unexpected device: {device}"
if device == "cpu":
    print("WARNING: MPS not available, training will be slow on CPU")

In [ ]:
# Cell 2: Data loading (Phase 1 outputs)
def load_jsonl(path):
    samples = []
    with open(path) as f:
        for line in f:
            samples.append(json.loads(line))
    return samples

# Load synthetic dataset (per D-06: identical splits for all candidates)
all_data = load_jsonl("../data/synthetic_scam_v1.jsonl")
train_data = [s for s in all_data if s["split"] == "train"]
val_data = [s for s in all_data if s["split"] == "val"]

# Label mapping: scam=1, safe=0
label_map = {"scam": 1, "safe": 0}
for s in train_data + val_data:
    s["label_id"] = label_map[s["label"]]

train_ds = Dataset.from_list(train_data)
val_ds = Dataset.from_list(val_data)

# Holdout (per D-07: primary evaluation metric)
holdout_data = load_jsonl("../data/holdout_realworld.jsonl")
for s in holdout_data:
    s["label_id"] = label_map[s["label"]]
holdout_ds = Dataset.from_list(holdout_data)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Holdout: {len(holdout_ds)}")
assert len(train_ds) > 15000, f"Expected >15K train samples, got {len(train_ds)}"
assert len(holdout_ds) >= 200, f"Expected >=200 holdout samples, got {len(holdout_ds)}"

In [ ]:
# Cell 3: Shared training configuration + tokenizer consistency check

# Per D-04: Fixed hyperparameters, no sweep
# Per D-05: Optimization deferred to Phase 4

BENCHMARK_CONFIG = {
    "num_train_epochs": 5,          # D-04: 3-5 range, using 5 for convergence
    "learning_rate": 2e-5,          # D-04: fixed
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 32,
    "max_length": 128,
    "fp16": False,                  # MPS does NOT support fp16 (Pitfall 4)
    "bf16": False,                  # MPS does NOT support bf16 (Pitfall 4)
}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", pos_label=1
    )
    return {"f1": f1, "precision": precision, "recall": recall}

# Architecture candidates (per D-10, D-11, D-12)
CANDIDATES = [
    {"name": "MobileBERT", "model_id": "google/mobilebert-uncased", "params_M": 24.6},
    {"name": "TinyBERT-4", "model_id": "huawei-noah/TinyBERT_General_4L_312D", "params_M": 14.4},
    {"name": "ELECTRA-small", "model_id": "google/electra-small-discriminator", "params_M": 13.5},
]

# DistilBERT explicitly excluded (per D-11: 66M params, over 50MB INT8 budget)
EXCLUDED = [
    {"name": "DistilBERT", "model_id": "distilbert-base-uncased", "reason": "66M params, exceeds 50MB INT8 budget (D-11)"}
]

print(f"Candidates: {[c['name'] for c in CANDIDATES]}")
print(f"Excluded: {[e['name'] for e in EXCLUDED]}")

# Tokenizer consistency assertion (Review Concern 3)
# All three architectures share the BERT 30,522-token WordPiece vocab.
EXPECTED_VOCAB_SIZE = 30522
print(f"\nTokenizer vocab size consistency check (expected: {EXPECTED_VOCAB_SIZE}):")
for candidate in CANDIDATES:
    tok = AutoTokenizer.from_pretrained(candidate["model_id"])
    actual = tok.vocab_size
    print(f"  {candidate['name']} ({candidate['model_id']}): vocab_size={actual}")
    assert actual == EXPECTED_VOCAB_SIZE, (
        f"TOKENIZER MISMATCH: {candidate['name']} has vocab_size={actual}, "
        f"expected {EXPECTED_VOCAB_SIZE}. This would break TextTokenizer.ts compatibility."
    )
    del tok
print("All tokenizers have identical vocab size -- input parity confirmed.")

In [ ]:
# Cell 4: Shared training function

def train_and_evaluate(model_id, model_name, train_ds, val_ds, holdout_ds, config):
    """Train a single architecture and return (trainer, tokenizer, val_results, holdout_results, train_result).

    If OOM occurs with batch_size=16 (likely for MobileBERT at 24.6M params on MPS),
    reduces to batch_size=8 and compensates with gradient_accumulation_steps=2
    to maintain effective batch size of 16.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)

    def tokenize_fn(examples):
        return tokenizer(
            examples["text"],
            padding="max_length",
            truncation=True,
            max_length=config["max_length"],
        )

    # Tokenize datasets
    train_tok = train_ds.map(tokenize_fn, batched=True).rename_column("label_id", "labels")
    val_tok = val_ds.map(tokenize_fn, batched=True).rename_column("label_id", "labels")
    holdout_tok = holdout_ds.map(tokenize_fn, batched=True).rename_column("label_id", "labels")

    # Determine columns to keep
    tok_cols = [c for c in ["input_ids", "attention_mask", "token_type_ids"] if c in train_tok.column_names]
    format_cols = tok_cols + ["labels"]

    train_tok.set_format("torch", columns=format_cols)
    val_tok.set_format("torch", columns=format_cols)
    holdout_tok.set_format("torch", columns=format_cols)

    output_dir = f"../models/benchmark_tmp/{model_name.lower().replace(' ', '_').replace('-', '_')}"

    # Determine batch size and gradient accumulation
    actual_batch_size = config["per_device_train_batch_size"]
    gradient_accumulation_steps = 1

    if model_name == "MobileBERT":
        # MobileBERT (24.6M params, 24 layers) needs reduced batch size on MPS
        actual_batch_size = 8
        gradient_accumulation_steps = 2
        print(f"  {model_name}: Using batch_size=8 + gradient_accumulation_steps=2 "
              f"(effective batch=16) to prevent MPS OOM")

    args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=config["num_train_epochs"],
        per_device_train_batch_size=actual_batch_size,
        per_device_eval_batch_size=config["per_device_eval_batch_size"],
        gradient_accumulation_steps=gradient_accumulation_steps,
        learning_rate=config["learning_rate"],
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        fp16=config["fp16"],
        bf16=config["bf16"],
        logging_steps=50,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        compute_metrics=compute_metrics,
    )

    # Train
    print(f"\n{'='*60}")
    print(f"Training {model_name} ({model_id})")
    print(f"  batch_size={actual_batch_size}, gradient_accumulation_steps={gradient_accumulation_steps}")
    print(f"  effective_batch_size={actual_batch_size * gradient_accumulation_steps}")
    print(f"{'='*60}")
    train_result = trainer.train()

    # Evaluate on validation set
    val_results = trainer.evaluate(val_tok)
    print(f"\n{model_name} Validation: F1={val_results['eval_f1']:.4f}, P={val_results['eval_precision']:.4f}, R={val_results['eval_recall']:.4f}")

    # Evaluate on holdout (PRIMARY METRIC per D-07)
    holdout_results = trainer.evaluate(holdout_tok, metric_key_prefix="holdout")
    print(f"{model_name} Holdout:    F1={holdout_results['holdout_f1']:.4f}, P={holdout_results['holdout_precision']:.4f}, R={holdout_results['holdout_recall']:.4f}")

    return trainer, tokenizer, val_results, holdout_results, train_result

In [ ]:
# Cell 5: MPS smoke test (1-epoch quick test)
print("Running MPS smoke test (TinyBERT-4, 1 epoch, 100 samples)...")

smoke_tokenizer = AutoTokenizer.from_pretrained("huawei-noah/TinyBERT_General_4L_312D")
smoke_model = AutoModelForSequenceClassification.from_pretrained(
    "huawei-noah/TinyBERT_General_4L_312D", num_labels=2
)

smoke_train = train_ds.select(range(min(100, len(train_ds))))
smoke_val = val_ds.select(range(min(50, len(val_ds))))

def smoke_tokenize(examples):
    return smoke_tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

smoke_train_tok = smoke_train.map(smoke_tokenize, batched=True).rename_column("label_id", "labels")
smoke_val_tok = smoke_val.map(smoke_tokenize, batched=True).rename_column("label_id", "labels")
smoke_train_tok.set_format("torch", columns=["input_ids", "attention_mask", "token_type_ids", "labels"])
smoke_val_tok.set_format("torch", columns=["input_ids", "attention_mask", "token_type_ids", "labels"])

smoke_args = TrainingArguments(
    output_dir="../models/benchmark_tmp/smoke_test",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    eval_strategy="epoch",
    fp16=False, bf16=False,
    logging_steps=10,
    report_to="none",
    save_strategy="no",
)

smoke_trainer = Trainer(
    model=smoke_model, args=smoke_args,
    train_dataset=smoke_train_tok, eval_dataset=smoke_val_tok,
    compute_metrics=compute_metrics,
)
smoke_result = smoke_trainer.train()
print(f"MPS smoke test PASSED. Training loss: {smoke_result.training_loss:.4f}")

# Cleanup
del smoke_model, smoke_trainer, smoke_train_tok, smoke_val_tok
if device == "mps":
    torch.mps.empty_cache()

In [ ]:
# Cell 6: Train MobileBERT
# NOTE: train_and_evaluate() automatically uses batch_size=8 + gradient_accumulation_steps=2
# for MobileBERT to prevent MPS OOM while maintaining effective batch size of 16
results_mobilebert = {}
trainer_mb, tokenizer_mb, val_mb, holdout_mb, train_mb = train_and_evaluate(
    "google/mobilebert-uncased", "MobileBERT",
    train_ds, val_ds, holdout_ds, BENCHMARK_CONFIG
)
results_mobilebert = {
    "val_f1": val_mb["eval_f1"],
    "holdout_f1": holdout_mb["holdout_f1"],
    "holdout_precision": holdout_mb["holdout_precision"],
    "holdout_recall": holdout_mb["holdout_recall"],
    "training_loss": train_mb.training_loss,
}
print(f"\nMobileBERT complete. Holdout F1: {results_mobilebert['holdout_f1']:.4f}")

if device == "mps":
    torch.mps.empty_cache()

In [ ]:
# Cell 7: Train TinyBERT-4
results_tinybert = {}
trainer_tb, tokenizer_tb, val_tb, holdout_tb, train_tb = train_and_evaluate(
    "huawei-noah/TinyBERT_General_4L_312D", "TinyBERT-4",
    train_ds, val_ds, holdout_ds, BENCHMARK_CONFIG
)
results_tinybert = {
    "val_f1": val_tb["eval_f1"],
    "holdout_f1": holdout_tb["holdout_f1"],
    "holdout_precision": holdout_tb["holdout_precision"],
    "holdout_recall": holdout_tb["holdout_recall"],
    "training_loss": train_tb.training_loss,
}
print(f"\nTinyBERT-4 complete. Holdout F1: {results_tinybert['holdout_f1']:.4f}")

if device == "mps":
    torch.mps.empty_cache()

In [ ]:
# Cell 8: Train ELECTRA-small
results_electra = {}
trainer_el, tokenizer_el, val_el, holdout_el, train_el = train_and_evaluate(
    "google/electra-small-discriminator", "ELECTRA-small",
    train_ds, val_ds, holdout_ds, BENCHMARK_CONFIG
)
results_electra = {
    "val_f1": val_el["eval_f1"],
    "holdout_f1": holdout_el["holdout_f1"],
    "holdout_precision": holdout_el["holdout_precision"],
    "holdout_recall": holdout_el["holdout_recall"],
    "training_loss": train_el.training_loss,
}
print(f"\nELECTRA-small complete. Holdout F1: {results_electra['holdout_f1']:.4f}")

if device == "mps":
    torch.mps.empty_cache()

In [ ]:
# Cell 9: Summary of training results
print("\n" + "="*80)
print("TRAINING RESULTS SUMMARY")
print("="*80)
for name, res in [("MobileBERT", results_mobilebert), ("TinyBERT-4", results_tinybert), ("ELECTRA-small", results_electra)]:
    print(f"\n{name}:")
    print(f"  Validation F1:  {res['val_f1']:.4f}")
    print(f"  Holdout F1:     {res['holdout_f1']:.4f}")
    print(f"  Holdout P/R:    {res['holdout_precision']:.4f} / {res['holdout_recall']:.4f}")
    print(f"  Training loss:  {res['training_loss']:.4f}")

print("\n" + "="*80)
print("RANKING (by Holdout F1 -- primary metric per D-07):")
print("="*80)
ranked = sorted(
    [("MobileBERT", results_mobilebert), ("TinyBERT-4", results_tinybert), ("ELECTRA-small", results_electra)],
    key=lambda x: x[1]["holdout_f1"],
    reverse=True,
)
for i, (name, res) in enumerate(ranked, 1):
    print(f"  {i}. {name}: Holdout F1={res['holdout_f1']:.4f}")

In [ ]:
# Cell 10: Confusion matrices for all 3 models on holdout
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, trainer_obj, tokenizer_obj) in enumerate([
    ("MobileBERT", trainer_mb, tokenizer_mb),
    ("TinyBERT-4", trainer_tb, tokenizer_tb),
    ("ELECTRA-small", trainer_el, tokenizer_el),
]):
    holdout_tok = holdout_ds.map(
        lambda ex: tokenizer_obj(ex["text"], padding="max_length", truncation=True, max_length=128),
        batched=True,
    ).rename_column("label_id", "labels")
    tok_cols = [c for c in ["input_ids", "attention_mask", "token_type_ids"] if c in holdout_tok.column_names]
    holdout_tok.set_format("torch", columns=tok_cols + ["labels"])

    preds_output = trainer_obj.predict(holdout_tok)
    preds = preds_output.predictions.argmax(axis=-1)
    labels = preds_output.label_ids

    cm = confusion_matrix(labels, preds)
    axes[idx].imshow(cm, cmap="Blues")
    axes[idx].set_title(f"{name}\nHoldout F1={f1_score(labels, preds, pos_label=1):.4f}")
    axes[idx].set_xlabel("Predicted")
    axes[idx].set_ylabel("Actual")
    axes[idx].set_xticks([0, 1])
    axes[idx].set_yticks([0, 1])
    axes[idx].set_xticklabels(["Safe", "Scam"])
    axes[idx].set_yticklabels(["Safe", "Scam"])
    for i in range(2):
        for j in range(2):
            axes[idx].text(j, i, str(cm[i][j]), ha="center", va="center",
                          color="white" if cm[i][j] > cm.max()/2 else "black", fontsize=14)

plt.tight_layout()
plt.savefig("../models/holdout_confusion_matrices.png", dpi=150)
plt.show()
print("Confusion matrices saved to research/models/holdout_confusion_matrices.png")

In [ ]:
# Cell 11: Per-architecture classification report on holdout
for name, trainer_obj, tokenizer_obj in [
    ("MobileBERT", trainer_mb, tokenizer_mb),
    ("TinyBERT-4", trainer_tb, tokenizer_tb),
    ("ELECTRA-small", trainer_el, tokenizer_el),
]:
    holdout_tok = holdout_ds.map(
        lambda ex: tokenizer_obj(ex["text"], padding="max_length", truncation=True, max_length=128),
        batched=True,
    ).rename_column("label_id", "labels")
    tok_cols = [c for c in ["input_ids", "attention_mask", "token_type_ids"] if c in holdout_tok.column_names]
    holdout_tok.set_format("torch", columns=tok_cols + ["labels"])
    preds_output = trainer_obj.predict(holdout_tok)
    preds = preds_output.predictions.argmax(axis=-1)
    labels = preds_output.label_ids
    print(f"\n{'='*60}")
    print(f"{name} -- Holdout Classification Report")
    print(f"{'='*60}")
    print(classification_report(labels, preds, target_names=["Safe", "Scam"]))